# FCE-Net: LOLv2 Cross-Dataset Evaluation & Inference
## Registration Number: 24BCE7695 | Experiment 2: Zero-Shot Cross-Dataset Benchmark

**Paper:** FCE-Net: A Frequency-Color-Edge Aware YUV Network for Low-Light Image Enhancement  
**Authors:** Zengyang Guo, Zhidan Cai, Haoyao Xia, Shifeng Hu  
**Venue:** Pattern Analysis and Applications (Springer), 2026  
**Official Repository:** https://github.com/gzy7/FCE-Net  
**Student Researcher:** Akhil Harshavardhan (`24BCE7695`)

---

### Experiment 2 Protocol
1. **Zero Training / Zero Fine-Tuning:** The model is evaluated strictly in zero-shot cross-dataset inference mode. No weights are modified or retrained.
2. **Checkpoint Provenance:** Reuses the verified `best_model.pth` trained from scratch on LOLv1 (300 epochs, Tesla T4, Best PSNR = 25.1846 dB).
3. **Dataset Scope:** Evaluates on the **LOLv2** benchmark (supporting `Real_captured` and `Synthetic` splits, up to 250 evaluation pairs).
4. **Target Outputs:**
   - Full-resolution enhanced images saved to `/kaggle/working/results_lolv2/enhanced/`
   - Quantitative evaluation report: PSNR (dB), SSIM, and LPIPS (AlexNet)
   - Visual comparison figure saved to `/kaggle/working/results_lolv2/comparison_lolv2.png`
   - Complete export package: `/kaggle/working/FCE_Net_LOLv2_Results.zip`

## 1. Environment Setup & Dependency Installation

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    for p in pkgs:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

try:
    import lpips
except ImportError:
    pip_install("lpips")

try:
    import pytorch_msssim
except ImportError:
    pip_install("pytorch_msssim")

try:
    import torchmetrics
except ImportError:
    pip_install("torchmetrics")

print("✅  All evaluation dependencies verified.")

## 2. Hardware / Device Check

In [ ]:
import torch
import os
import warnings

warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    DEVICE = torch.device("cpu")
    print("⚠️ No GPU detected — evaluation will run on CPU.")
    print("   Recommendation: Enable Kaggle GPU accelerator for faster inference.")

print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

## 3. Source Code Setup (Inline FCE-Net Architecture)
All neural modules are defined inline for 100% self-contained Kaggle execution.
The architecture matches the LOLv1 training experiment (686,076 parameters).

### 3a. Wavelet Transform (DWT / IDWT)

In [ ]:
# ── wavelet_transform.py ─────────────────────────────────────────────
# Renamed from wave.py to avoid conflict with Python stdlib wave module.
import pywt
import torch
import torch.nn as nn
from torch.autograd import Function

class DWT_Function(Function):
    @staticmethod
    def forward(ctx, x, w_ll, w_lh, w_hl, w_hh):
        x = x.contiguous()
        ctx.save_for_backward(w_ll, w_lh, w_hl, w_hh)
        ctx.shape = x.shape
        dim = x.shape[1]
        x_ll = torch.nn.functional.conv2d(x, w_ll.expand(dim, -1, -1, -1), stride=2, groups=dim)
        x_lh = torch.nn.functional.conv2d(x, w_lh.expand(dim, -1, -1, -1), stride=2, groups=dim)
        x_hl = torch.nn.functional.conv2d(x, w_hl.expand(dim, -1, -1, -1), stride=2, groups=dim)
        x_hh = torch.nn.functional.conv2d(x, w_hh.expand(dim, -1, -1, -1), stride=2, groups=dim)
        x = torch.cat([x_ll, x_lh, x_hl, x_hh], dim=1)
        return x

    @staticmethod
    def backward(ctx, dx):
        if ctx.needs_input_grad[0]:
            w_ll, w_lh, w_hl, w_hh = ctx.saved_tensors
            B, C, H, W = ctx.shape
            dx = dx.view(B, 4, -1, H // 2, W // 2)
            dx = dx.transpose(1, 2).reshape(B, -1, H // 2, W // 2)
            filters = torch.cat([w_ll, w_lh, w_hl, w_hh], dim=0).repeat(C, 1, 1, 1)
            dx = torch.nn.functional.conv_transpose2d(dx, filters, stride=2, groups=C)
        return dx, None, None, None, None

class IDWT_Function(Function):
    @staticmethod
    def forward(ctx, x, filters):
        ctx.save_for_backward(filters)
        ctx.shape = x.shape
        B, _, H, W = x.shape
        x = x.view(B, 4, -1, H, W).transpose(1, 2)
        C = x.shape[1]
        x = x.reshape(B, -1, H, W)
        filters = filters.repeat(C, 1, 1, 1)
        x = torch.nn.functional.conv_transpose2d(x, filters, stride=2, groups=C)
        return x

    @staticmethod
    def backward(ctx, dx):
        if ctx.needs_input_grad[0]:
            filters = ctx.saved_tensors[0]
            B, C, H, W = ctx.shape
            C = C // 4
            dx = dx.contiguous()
            w_ll, w_lh, w_hl, w_hh = torch.unbind(filters, dim=0)
            x_ll = torch.nn.functional.conv2d(dx, w_ll.unsqueeze(1).expand(C, -1, -1, -1), stride=2, groups=C)
            x_lh = torch.nn.functional.conv2d(dx, w_lh.unsqueeze(1).expand(C, -1, -1, -1), stride=2, groups=C)
            x_hl = torch.nn.functional.conv2d(dx, w_hl.unsqueeze(1).expand(C, -1, -1, -1), stride=2, groups=C)
            x_hh = torch.nn.functional.conv2d(dx, w_hh.unsqueeze(1).expand(C, -1, -1, -1), stride=2, groups=C)
            dx = torch.cat([x_ll, x_lh, x_hl, x_hh], dim=1)
        return dx, None

class IDWT_2D(nn.Module):
    def __init__(self, wave):
        super(IDWT_2D, self).__init__()
        w = pywt.Wavelet(wave)
        rec_hi = torch.Tensor(w.rec_hi)
        rec_lo = torch.Tensor(w.rec_lo)
        w_ll = rec_lo.unsqueeze(0) * rec_lo.unsqueeze(1)
        w_lh = rec_lo.unsqueeze(0) * rec_hi.unsqueeze(1)
        w_hl = rec_hi.unsqueeze(0) * rec_lo.unsqueeze(1)
        w_hh = rec_hi.unsqueeze(0) * rec_hi.unsqueeze(1)
        w_ll = w_ll.unsqueeze(0).unsqueeze(1)
        w_lh = w_lh.unsqueeze(0).unsqueeze(1)
        w_hl = w_hl.unsqueeze(0).unsqueeze(1)
        w_hh = w_hh.unsqueeze(0).unsqueeze(1)
        filters = torch.cat([w_ll, w_lh, w_hl, w_hh], dim=0)
        self.register_buffer('filters', filters.to(dtype=torch.float32))

    def forward(self, x):
        return IDWT_Function.apply(x, self.filters)


class DWT_2D(nn.Module):
    def __init__(self, wave):
        super(DWT_2D, self).__init__()
        w = pywt.Wavelet(wave)
        dec_hi = torch.Tensor(w.dec_hi[::-1])
        dec_lo = torch.Tensor(w.dec_lo[::-1])
        w_ll = dec_lo.unsqueeze(0) * dec_lo.unsqueeze(1)
        w_lh = dec_lo.unsqueeze(0) * dec_hi.unsqueeze(1)
        w_hl = dec_hi.unsqueeze(0) * dec_lo.unsqueeze(1)
        w_hh = dec_hi.unsqueeze(0) * dec_hi.unsqueeze(1)
        self.register_buffer('w_ll', w_ll.unsqueeze(0).unsqueeze(0).to(dtype=torch.float32))
        self.register_buffer('w_lh', w_lh.unsqueeze(0).unsqueeze(0).to(dtype=torch.float32))
        self.register_buffer('w_hl', w_hl.unsqueeze(0).unsqueeze(0).to(dtype=torch.float32))
        self.register_buffer('w_hh', w_hh.unsqueeze(0).unsqueeze(0).to(dtype=torch.float32))

    def forward(self, x):
        return DWT_Function.apply(x, self.w_ll, self.w_lh, self.w_hl, self.w_hh)

print("✅  Wavelet Transform modules defined (DWT_2D, IDWT_2D)")

### 3b. HLFDB — High-Low Frequency Decomposition Block

In [ ]:
# ── HLFDB.py ─────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
import numbers

class CALayer(nn.Module):
    def __init__(self, channel, reduction=16):
        super(CALayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_du = nn.Sequential(
            nn.Conv2d(channel, channel // reduction, 1, padding=0, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(channel // reduction, channel, 1, padding=0, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        y = self.avg_pool(x)
        y = self.conv_du(y)
        return x * y

class Dense(nn.Module):
    def __init__(self, in_channels):
        super(Dense, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, stride=1)
        self.conv2 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, stride=1)
        self.conv3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, stride=1)
        self.conv4 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, stride=1)
        self.conv5 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, stride=1)
        self.conv6 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, stride=1)
        self.gelu = nn.GELU()

    def forward(self, x):
        x1 = self.gelu(self.conv1(x) + x)
        x2 = self.gelu(self.conv2(x1) + x1 + x)
        x3 = self.gelu(self.conv3(x2) + x2 + x1 + x)
        x4 = self.gelu(self.conv4(x3) + x3 + x2 + x1 + x)
        x5 = self.gelu(self.conv5(x4) + x4 + x3 + x2 + x1 + x)
        x6 = self.gelu(self.conv6(x5) + x5 + x4 + x3 + x2 + x1 + x)
        return x6

class ResNet_Block(nn.Module):
    def __init__(self, in_channels):
        super(ResNet_Block, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)

    def forward(self, x):
        out1 = F.gelu(self.conv1(x))
        out2 = F.gelu(self.conv2(out1))
        out2 += x
        return out2

class Fusion(nn.Module):
    def __init__(self, in_channels, wave):
        super(Fusion, self).__init__()
        self.dwt = DWT_2D(wave)
        self.convh1 = nn.Conv2d(in_channels * 3, in_channels, kernel_size=1, stride=1, padding=0, bias=True)
        self.high = ResNet_Block(in_channels)
        self.convh2 = nn.Conv2d(in_channels, in_channels * 3, kernel_size=1, stride=1, padding=0, bias=True)
        self.convl = nn.Conv2d(in_channels * 2, in_channels, kernel_size=1, stride=1, padding=0, bias=True)
        self.low = ResNet_Block(in_channels)
        self.idwt = IDWT_2D(wave)

    def forward(self, x1, x2):
        b, c, h, w = x1.shape
        x_dwt = self.dwt(x1)
        ll, lh, hl, hh = x_dwt.split(c, 1)
        high = torch.cat([lh, hl, hh], 1)
        high1 = self.convh1(high)
        high2 = self.high(high1)
        highf = self.convh2(high2)
        b1, c1, h1, w1 = ll.shape
        b2, c2, h2, w2 = x2.shape
        if h1 != h2:
            x2 = F.pad(x2, (0, 0, 1, 0), "constant", 0)
        low = torch.cat([ll, x2], 1)
        low = self.convl(low)
        lowf = self.low(low)
        out = torch.cat((lowf, highf), 1)
        out_idwt = self.idwt(out)
        return out_idwt

# ── LayerNorm variants ──
def to_3d(x):
    return rearrange(x, 'b c h w -> b (h w) c')

def to_4d(x, h, w):
    return rearrange(x, 'b (h w) c -> b c h w', h=h, w=w)

class BiasFree_LayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super(BiasFree_LayerNorm, self).__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        normalized_shape = torch.Size(normalized_shape)
        assert len(normalized_shape) == 1
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.normalized_shape = normalized_shape

    def forward(self, x):
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return x / torch.sqrt(sigma + 1e-5) * self.weight

class WithBias_LayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super(WithBias_LayerNorm, self).__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        normalized_shape = torch.Size(normalized_shape)
        assert len(normalized_shape) == 1
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.normalized_shape = normalized_shape

    def forward(self, x):
        mu = x.mean(-1, keepdim=True)
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return (x - mu) / torch.sqrt(sigma + 1e-5) * self.weight + self.bias

class HLFDB_LayerNorm(nn.Module):
    def __init__(self, dim, LayerNorm_type):
        super(HLFDB_LayerNorm, self).__init__()
        if LayerNorm_type == 'BiasFree':
            self.body = BiasFree_LayerNorm(dim)
        else:
            self.body = WithBias_LayerNorm(dim)

    def forward(self, x):
        h, w = x.shape[-2:]
        return to_4d(self.body(to_3d(x)), h, w)

class FeedForward(nn.Module):
    def __init__(self, dim, ffn_expansion_factor, bias):
        super(FeedForward, self).__init__()
        hidden_features = int(dim * ffn_expansion_factor)
        self.project_in = nn.Conv2d(dim, hidden_features * 2, kernel_size=1, bias=bias)
        self.dwconv = nn.Conv2d(hidden_features * 2, hidden_features * 2, kernel_size=3,
                                stride=1, padding=1, groups=hidden_features * 2, bias=bias)
        self.project_out = nn.Conv2d(hidden_features, dim, kernel_size=1, bias=bias)

    def forward(self, x):
        x = self.project_in(x)
        x1, x2 = self.dwconv(x).chunk(2, dim=1)
        x = F.gelu(x1) * x2
        x = self.project_out(x)
        return x

class TransposedAttention(nn.Module):
    def __init__(self, dim, num_heads, bias):
        super(TransposedAttention, self).__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        self.qkv = nn.Conv2d(dim, dim * 3, kernel_size=1, bias=bias)
        self.qkv_dwconv = nn.Conv2d(dim * 3, dim * 3, kernel_size=3, stride=1, padding=1,
                                     groups=dim * 3, bias=bias)
        self.project_out = nn.Conv2d(dim, dim, kernel_size=1, bias=bias)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.qkv_dwconv(self.qkv(x))
        q, k, v = qkv.chunk(3, dim=1)
        q = rearrange(q, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        k = rearrange(k, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        v = rearrange(v, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        q = torch.nn.functional.normalize(q, dim=-1)
        k = torch.nn.functional.normalize(k, dim=-1)
        attn = (q @ k.transpose(-2, -1)) * self.temperature
        attn = attn.softmax(dim=-1)
        out = (attn @ v)
        out = rearrange(out, 'b head c (h w) -> b (head c) h w', head=self.num_heads, h=h, w=w)
        out = self.project_out(out)
        return out

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, ffn_expansion_factor, bias, LayerNorm_type):
        super(TransformerBlock, self).__init__()
        self.norm1 = HLFDB_LayerNorm(dim, LayerNorm_type)
        self.attn = TransposedAttention(dim, num_heads, bias)
        self.norm2 = HLFDB_LayerNorm(dim, LayerNorm_type)
        self.ffn = FeedForward(dim, ffn_expansion_factor, bias)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class UNet_HLFDB(nn.Module):
    def __init__(self, in_channels, wave):
        super(UNet_HLFDB, self).__init__()
        self.trans1 = TransformerBlock(in_channels, 8, 2.66, False, 'WithBias')
        self.trans2 = TransformerBlock(in_channels, 8, 2.66, False, 'WithBias')
        self.trans3 = TransformerBlock(in_channels, 8, 2.66, False, 'WithBias')
        self.avgpool1 = nn.AvgPool2d(kernel_size=2)
        self.avgpool2 = nn.AvgPool2d(kernel_size=2)
        self.upsample1 = Fusion(in_channels, wave)
        self.upsample2 = Fusion(in_channels, wave)

    def forward(self, x):
        x1 = x
        x1_r = self.trans1(x)
        x2 = self.avgpool1(x1)
        x2_r = self.trans2(x2)
        x3 = self.avgpool2(x2)
        x3_r = self.trans3(x3)
        x4 = self.upsample1(x2_r, x3_r)
        out = self.upsample2(x1_r, x4)
        b1, c1, h1, w1 = out.shape
        b2, c2, h2, w2 = x.shape
        if h1 != h2:
            out = F.pad(out, (0, 0, 1, 0), "constant", 0)
        return out + x

class HPB(nn.Module):
    def __init__(self, n_feats, wave):
        super(HPB, self).__init__()
        self.down = nn.AvgPool2d(kernel_size=2)
        self.dense = Dense(n_feats)
        self.unet = UNet_HLFDB(n_feats, wave)
        self.alise1 = nn.Conv2d(2 * n_feats, n_feats, 1, 1, 0)
        self.alise2 = nn.Conv2d(n_feats, n_feats, 3, 1, 1)
        self.att = CALayer(n_feats)

    def forward(self, x):
        low = self.down(x)
        high = x - F.interpolate(low, size=x.size()[-2:], mode='bilinear', align_corners=True)
        lowf = self.unet(low)
        highfeat = self.dense(high)
        lowfeat = F.interpolate(lowf, size=x.size()[-2:], mode='bilinear', align_corners=True)
        out = self.alise2(self.att(self.alise1(torch.cat([highfeat, lowfeat], dim=1)))) + x
        return out

class UN(nn.Module):
    """Top-level High-Low Frequency Decomposition Module (HFDM)."""
    def __init__(self, n_feats):
        super(UN, self).__init__()
        self.encoder1 = HPB(n_feats, 'haar')
        self.encoder2 = HPB(n_feats, 'haar')
        self.encoder3 = HPB(n_feats, 'haar')
        self.conv = nn.Conv2d(n_feats * 3, n_feats, kernel_size=1)

    def forward(self, x):
        x1 = self.encoder1(x)
        x2 = self.encoder2(x1)
        x3 = self.encoder3(x2)
        out = torch.cat((x1, x2, x3), 1)
        out = self.conv(out)
        return out

print("✅  HLFDB modules defined (UN, HPB, UNet, TransformerBlock, Fusion, Dense, CALayer)")

### 3c. FCE-Net Model

In [ ]:
# ── model.py ─────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init

class LayerNormalization(nn.Module):
    def __init__(self, dim):
        super(LayerNormalization, self).__init__()
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        return x.permute(0, 3, 1, 2)

class SEBlock(nn.Module):
    def __init__(self, input_channels, reduction_ratio=16):
        super(SEBlock, self).__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(input_channels, input_channels // reduction_ratio)
        self.fc2 = nn.Linear(input_channels // reduction_ratio, input_channels)
        self._init_weights()

    def forward(self, x):
        batch_size, num_channels, _, _ = x.size()
        y = self.pool(x).reshape(batch_size, num_channels)
        y = F.relu(self.fc1(y))
        y = torch.tanh(self.fc2(y))
        y = y.reshape(batch_size, num_channels, 1, 1)
        return x * y

    def _init_weights(self):
        init.kaiming_uniform_(self.fc1.weight, a=0, mode='fan_in', nonlinearity='relu')
        init.kaiming_uniform_(self.fc2.weight, a=0, mode='fan_in', nonlinearity='relu')
        init.constant_(self.fc1.bias, 0)
        init.constant_(self.fc2.bias, 0)

class MSEFBlock(nn.Module):
    def __init__(self, filters):
        super(MSEFBlock, self).__init__()
        self.layer_norm = LayerNormalization(filters)
        self.depthwise_conv = nn.Conv2d(filters, filters, kernel_size=3, padding=1, groups=filters)
        self.se_attn = SEBlock(filters)
        self._init_weights()

    def forward(self, x):
        x_norm = self.layer_norm(x)
        x1 = self.depthwise_conv(x_norm)
        x2 = self.se_attn(x_norm)
        x_fused = x1 * x2
        x_out = x_fused + x
        return x_out

    def _init_weights(self):
        init.kaiming_uniform_(self.depthwise_conv.weight, a=0, mode='fan_in', nonlinearity='relu')
        init.constant_(self.depthwise_conv.bias, 0)

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_size, num_heads):
        super(MultiHeadSelfAttention, self).__init__()
        self.embed_size = embed_size
        self.num_heads = num_heads
        assert embed_size % num_heads == 0
        self.head_dim = embed_size // num_heads
        self.query_dense = nn.Linear(embed_size, embed_size)
        self.key_dense = nn.Linear(embed_size, embed_size)
        self.value_dense = nn.Linear(embed_size, embed_size)
        self.combine_heads = nn.Linear(embed_size, embed_size)
        self._init_weights()

    def split_heads(self, x, batch_size):
        x = x.reshape(batch_size, -1, self.num_heads, self.head_dim)
        return x.permute(0, 2, 1, 3)

    def forward(self, x):
        batch_size, _, height, width = x.size()
        x = x.reshape(batch_size, height * width, -1)
        query = self.split_heads(self.query_dense(x), batch_size)
        key = self.split_heads(self.key_dense(x), batch_size)
        value = self.split_heads(self.value_dense(x), batch_size)
        attention_weights = F.softmax(
            torch.matmul(query, key.transpose(-2, -1)) / (self.head_dim ** 0.5), dim=-1)
        attention = torch.matmul(attention_weights, value)
        attention = attention.permute(0, 2, 1, 3).contiguous().reshape(
            batch_size, -1, self.embed_size)
        output = self.combine_heads(attention)
        return output.reshape(batch_size, height, width, self.embed_size).permute(0, 3, 1, 2)

    def _init_weights(self):
        init.xavier_uniform_(self.query_dense.weight)
        init.xavier_uniform_(self.key_dense.weight)
        init.xavier_uniform_(self.value_dense.weight)
        init.xavier_uniform_(self.combine_heads.weight)
        init.constant_(self.query_dense.bias, 0)
        init.constant_(self.key_dense.bias, 0)
        init.constant_(self.value_dense.bias, 0)
        init.constant_(self.combine_heads.bias, 0)

class Denoiser(nn.Module):
    def __init__(self, num_filters, kernel_size=3, activation='relu'):
        super(Denoiser, self).__init__()
        self.conv1 = nn.Conv2d(1, num_filters, kernel_size=kernel_size, padding=1)
        self.conv2 = nn.Conv2d(num_filters, num_filters, kernel_size=kernel_size, stride=2, padding=1)
        self.conv3 = nn.Conv2d(num_filters, num_filters, kernel_size=kernel_size, stride=2, padding=1)
        self.conv4 = nn.Conv2d(num_filters, num_filters, kernel_size=kernel_size, stride=2, padding=1)
        self.bottleneck = MultiHeadSelfAttention(embed_size=num_filters, num_heads=4)
        self.up2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.up3 = nn.Upsample(scale_factor=2, mode='nearest')
        self.up4 = nn.Upsample(scale_factor=2, mode='nearest')
        self.output_layer = nn.Conv2d(1, 1, kernel_size=kernel_size, padding=1)
        self.res_layer = nn.Conv2d(num_filters, 1, kernel_size=kernel_size, padding=1)
        self.activation = getattr(F, activation)
        self._init_weights()

    def forward(self, x):
        x1 = self.activation(self.conv1(x))
        x2 = self.activation(self.conv2(x1))
        x3 = self.activation(self.conv3(x2))
        x4 = self.activation(self.conv4(x3))
        x = self.bottleneck(x4)
        x = self.up4(x)
        x = self.up3(x + x3)
        x = self.up2(x + x2)
        x = x + x1
        x = self.res_layer(x)
        return torch.tanh(self.output_layer(x + x))

    def _init_weights(self):
        for layer in [self.conv1, self.conv2, self.conv3, self.conv4,
                      self.output_layer, self.res_layer]:
            init.kaiming_uniform_(layer.weight, a=0, mode='fan_in', nonlinearity='relu')
            if layer.bias is not None:
                init.constant_(layer.bias, 0)

class ColorAttentionGuidance(nn.Module):
    def __init__(self, in_channels=2, reduction=8):
        super(ColorAttentionGuidance, self).__init__()
        inter_channels = max(1, in_channels // reduction)
        self.conv1 = nn.Conv2d(in_channels, inter_channels, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(inter_channels, in_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        attention = self.conv1(x)
        attention = self.relu(attention)
        attention = self.conv2(attention)
        attention_map = self.sigmoid(attention)
        return x * attention_map

class EdgeEnhance(nn.Module):
    def __init__(self, dim, enabled=False):
        super().__init__()
        self.enabled = enabled
        self.edge_branch = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(dim, dim, kernel_size=3, padding=2, dilation=2)
        )
        self.high_freq_branch = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=1),
            nn.GELU(),
            nn.Conv2d(dim, dim, kernel_size=3, padding=1)
        )
        self.fusion = nn.Conv2d(dim * 2, dim, kernel_size=1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        if not self.enabled:
            return x
        edge_feat = self.edge_branch(x)
        high_freq_feat = self.high_freq_branch(x)
        enhanced = self.fusion(torch.cat([edge_feat, high_freq_feat], dim=1))
        return x + self.gamma * enhanced

class FCE(nn.Module):
    """FCE-Net main model — Frequency-Color-Edge aware YUV network."""
    def __init__(self, filters=32):
        super(FCE, self).__init__()
        self.process_y = self._create_processing_layers(filters)
        self.process_cb = self._create_processing_layers(filters)
        self.process_cr = self._create_processing_layers(filters)
        self.hfdm = UN(filters)
        self.denoiser_cb = Denoiser(filters // 2)
        self.denoiser_cr = Denoiser(filters // 2)
        self.cagm = ColorAttentionGuidance(in_channels=64)
        self.edge_enhance = EdgeEnhance(dim=filters, enabled=True)
        self.lum_pool = nn.MaxPool2d(8)
        self.lum_mhsa = MultiHeadSelfAttention(embed_size=filters, num_heads=4)
        self.lum_up = nn.Upsample(scale_factor=8, mode='nearest')
        self.lum_conv = nn.Conv2d(filters, filters, kernel_size=1, padding=0)
        self.ref_conv = nn.Conv2d(filters * 2, filters, kernel_size=1, padding=0)
        self.msef = MSEFBlock(filters)
        self.recombine = nn.Conv2d(filters * 2, filters, kernel_size=3, padding=1)
        self.final_adjustments = nn.Conv2d(filters, 3, kernel_size=3, padding=1)
        self._init_weights()

    def _create_processing_layers(self, filters):
        return nn.Sequential(
            nn.Conv2d(1, filters, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def _rgb_to_ycbcr(self, image):
        r, g, b = image[:, 0, :, :], image[:, 1, :, :], image[:, 2, :, :]
        y = 0.299 * r + 0.587 * g + 0.114 * b
        u = -0.14713 * r - 0.28886 * g + 0.436 * b + 0.5
        v = 0.615 * r - 0.51499 * g - 0.10001 * b + 0.5
        yuv = torch.stack((y, u, v), dim=1)
        return yuv

    def forward(self, inputs):
        ycbcr = self._rgb_to_ycbcr(inputs)
        y, cb, cr = torch.split(ycbcr, 1, dim=1)
        cb = self.denoiser_cb(cb) + cb
        cr = self.denoiser_cr(cr) + cr
        y_feat = self.process_y(y)
        y_processed = self.hfdm(y_feat)
        cb_processed = self.process_cb(cb)
        cr_processed = self.process_cr(cr)
        color = torch.cat([cb_processed, cr_processed], dim=1)
        ref = self.cagm(color)
        y_processed = self.edge_enhance(y_processed)
        lum = y_processed
        lum_1 = self.lum_pool(lum)
        lum_1 = self.lum_mhsa(lum_1)
        lum_1 = self.lum_up(lum_1)
        lum = lum + lum_1
        ref = self.ref_conv(ref)
        shortcut = ref
        ref = ref + 0.2 * self.lum_conv(lum)
        ref = self.msef(ref)
        ref = ref + shortcut
        recombined = self.recombine(torch.cat([ref, lum], dim=1))
        output = self.final_adjustments(recombined)
        return torch.sigmoid(output)

    def _init_weights(self):
        for module in self.children():
            if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
                init.kaiming_uniform_(module.weight, a=0, mode='fan_in', nonlinearity='relu')
                if module.bias is not None:
                    init.constant_(module.bias, 0)

print("✅  FCE model defined")

### 3d. Evaluation Dataset & Image Handling Utilities

In [ ]:
import os
import glob
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

class LOLv2EvalDataset(Dataset):
    """Dataset for LOLv2 evaluation (supports both paired and unpaired testing)."""
    def __init__(self, low_paths, high_paths=None, max_images=None):
        self.low_paths = sorted(low_paths)
        self.high_paths = sorted(high_paths) if high_paths else None
        
        if self.high_paths:
            n = min(len(self.low_paths), len(self.high_paths))
            self.low_paths = self.low_paths[:n]
            self.high_paths = self.high_paths[:n]
            
        if max_images and max_images > 0:
            self.low_paths = self.low_paths[:max_images]
            if self.high_paths:
                self.high_paths = self.high_paths[:max_images]

        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.low_paths)

    def __getitem__(self, idx):
        low_p = self.low_paths[idx]
        low_img = Image.open(low_p).convert('RGB')
        low_tensor = self.to_tensor(low_img)
        fname = os.path.basename(low_p)

        if self.high_paths:
            high_p = self.high_paths[idx]
            high_img = Image.open(high_p).convert('RGB')
            high_tensor = self.to_tensor(high_img)
            return low_tensor, high_tensor, fname
        
        return low_tensor, fname

print("✅  Evaluation dataset class defined.")

## 4. LOLv1 Trained Checkpoint Auto-Detection & Verification
Locates the trained checkpoint from your previous LOLv1 Kaggle run.

**How to connect in Kaggle:**
1. In the right sidebar, click **"+ Add Input"** (or "Add Data").
2. Select **"Notebook Output Files"**.
3. Search for your previous LOLv1 notebook name (e.g. `fce-net-kaggle` or `notebook356021ef66`).
4. Click **"Add"**. Kaggle will mount the output under `/kaggle/input/<notebook-name>/`.
5. Alternatively, if you uploaded `best_model.pth` or `FCE_Net_Results.zip` as a dataset, it will be auto-detected.

In [ ]:
import os
import glob
import zipfile
import torch

CHECKPOINT_PATH = None
EXTRACTED_DIR = "/kaggle/working/imported_checkpoint"

# 1. Search for extracted checkpoint files (.pth)
search_patterns = [
    "/kaggle/input/**/best_model.pth",
    "/kaggle/input/**/final_model.pth",
    "/kaggle/input/**/checkpoint_epoch*.pth",
    "/kaggle/input/**/*model*.pth",
    "/kaggle/working/checkpoints/best_model.pth",
    "/kaggle/working/**/best_model.pth",
    "./**/*best_model*.pth",
    "./**/*model*.pth"
]

found_pths = []
for pattern in search_patterns:
    for p in glob.glob(pattern, recursive=True):
        if os.path.isfile(p) and not p.endswith("vgg19.pth"):
            found_pths.append(p)

# 2. Search for exported ZIP archive from previous run
if not found_pths:
    zip_patterns = [
        "/kaggle/input/**/FCE_Net_Results*.zip",
        "/kaggle/input/**/*Results*.zip",
        "/kaggle/input/**/*checkpoint*.zip",
        "/kaggle/working/FCE_Net_Results.zip",
        "./*.zip"
    ]
    for pattern in zip_patterns:
        for z in glob.glob(pattern, recursive=True):
            if os.path.isfile(z):
                print(f"📦 Found checkpoint archive: {z}")
                os.makedirs(EXTRACTED_DIR, exist_ok=True)
                with zipfile.ZipFile(z, 'r') as zip_ref:
                    zip_ref.extractall(EXTRACTED_DIR)
                print(f"   Extracted to {EXTRACTED_DIR}")
                for p in glob.glob(f"{EXTRACTED_DIR}/**/*.pth", recursive=True):
                    found_pths.append(p)
                break

# 3. Validate candidates against FCE-Net architecture
model_candidate = FCE(filters=32)
expected_keys = set(model_candidate.state_dict().keys())
verified_checkpoint = None

for candidate in found_pths:
    try:
        ckpt = torch.load(candidate, map_location="cpu")
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        elif isinstance(ckpt, dict):
            state_dict = ckpt
        else:
            continue
            
        common_keys = expected_keys.intersection(set(state_dict.keys()))
        if len(common_keys) >= len(expected_keys) * 0.9:
            verified_checkpoint = candidate
            CHECKPOINT_STATE = state_dict
            print(f"✅ VERIFIED LOLv1 CHECKPOINT: {candidate}")
            print(f"   Architecture match: {len(common_keys)}/{len(expected_keys)} tensors")
            print(f"   File size: {os.path.getsize(candidate) / 1e6:.2f} MB")
            break
    except Exception as e:
        continue

if verified_checkpoint:
    CHECKPOINT_PATH = verified_checkpoint
else:
    print("=" * 68)
    print("ℹ️  NO ATTACHED CHECKPOINT DETECTED YET")
    print("=" * 68)
    print("To attach your LOLv1 checkpoint in Kaggle:")
    print("  1. In the right panel, click '+ Add Input'")
    print("  2. Select 'Notebook Output Files'")
    print("  3. Find your previous executed notebook and click 'Add'")
    print("  4. Or click '+ Add Input' -> 'Upload Dataset' -> upload best_model.pth")
    print("=" * 68)

## 5. LOLv2 Dataset Auto-Discovery & Validation
Auto-scans `/kaggle/input/` for the LOLv2 dataset (Real_captured and/or Synthetic test splits).

In [ ]:
import os
import glob
from PIL import Image

# Maximum evaluation images to process (set to 250 as requested, or None for all)
MAX_EVAL_IMAGES = 250

LOLV2_TEST_LOW = None
LOLV2_TEST_HIGH = None
DATASET_SPLIT_NAME = "Unknown"

# Search candidates for LOLv2
search_dirs = [
    # LOLv2 Real captured
    ("LOLv2 Real-Captured Test",
     "/kaggle/input/**/LOLv2/Real_captured/Test/Low",
     "/kaggle/input/**/LOLv2/Real_captured/Test/Normal"),
    ("LOLv2 Real-Captured (lowercase)",
     "/kaggle/input/**/real_captured/test/low",
     "/kaggle/input/**/real_captured/test/normal"),
    ("LOLv2 Real Test",
     "/kaggle/input/**/Real/Test/Low",
     "/kaggle/input/**/Real/Test/Normal"),
    # LOLv2 Synthetic
    ("LOLv2 Synthetic Test",
     "/kaggle/input/**/LOLv2/Synthetic/Test/Low",
     "/kaggle/input/**/LOLv2/Synthetic/Test/Normal"),
    ("LOLv2 Synthetic (lowercase)",
     "/kaggle/input/**/synthetic/test/low",
     "/kaggle/input/**/synthetic/test/normal"),
    # Generic Kaggle low/high test folders
    ("Generic LOLv2 / Low-Light Test",
     "/kaggle/input/**/test/low",
     "/kaggle/input/**/test/high"),
    ("Generic LOLv2 Low/Normal",
     "/kaggle/input/**/test/low",
     "/kaggle/input/**/test/normal"),
    ("LOLv2 our / eval",
     "/kaggle/input/**/eval*/low",
     "/kaggle/input/**/eval*/high")
]

for name, low_pat, high_pat in search_dirs:
    low_matches = glob.glob(low_pat, recursive=True)
    high_matches = glob.glob(high_pat, recursive=True)
    if low_matches:
        cand_low = [d for d in low_matches if os.path.isdir(d)]
        cand_high = [d for d in high_matches if os.path.isdir(d)]
        if cand_low:
            LOLV2_TEST_LOW = cand_low[0]
            DATASET_SPLIT_NAME = name
            if cand_high:
                LOLV2_TEST_HIGH = cand_high[0]
            break

# If not found on Kaggle, check local paths or prepare synthetic audit pairs
if not LOLV2_TEST_LOW or not os.path.isdir(str(LOLV2_TEST_LOW)):
    local_cands = glob.glob("./**/Test/Low", recursive=True) + glob.glob("./**/test/low", recursive=True)
    if local_cands:
        LOLV2_TEST_LOW = local_cands[0]
        DATASET_SPLIT_NAME = "Local Test Split"
        high_cands = glob.glob("./**/Test/Normal", recursive=True) + glob.glob("./**/test/high", recursive=True)
        if high_cands:
            LOLV2_TEST_HIGH = high_cands[0]

# Display discovery summary
print("=" * 68)
print("LOLv2 DATASET DISCOVERY REPORT")
print("=" * 68)
if LOLV2_TEST_LOW:
    low_imgs = [f for f in os.listdir(LOLV2_TEST_LOW) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"✅ Split Name      : {DATASET_SPLIT_NAME}")
    print(f"   Low-light Path  : {LOLV2_TEST_LOW} ({len(low_imgs)} images)")
    if LOLV2_TEST_HIGH:
        high_imgs = [f for f in os.listdir(LOLV2_TEST_HIGH) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        print(f"   Ground Truth    : {LOLV2_TEST_HIGH} ({len(high_imgs)} images)")
        print(f"   Evaluation Mode : Paired Benchmark (PSNR + SSIM + LPIPS)")
    else:
        print(f"   Ground Truth    : None detected (Unpaired inference mode)")
    print(f"   Target Limit    : {MAX_EVAL_IMAGES} images")
else:
    print("ℹ️  No LOLv2 dataset attached yet.")
    print("   To attach LOLv2 in Kaggle:")
    print("     1. Click '+ Add Input'")
    print("     2. Search 'LOL-v2' or 'LOLv2 dataset'")
    print("     3. Click 'Add'")
    print("   A 3-sample synthetic audit set will be used for smoke verification.")
print("=" * 68)

## 6. Startup Validation (Pre-Flight Input Check)
Guarantees that both required inputs (LOLv1 Checkpoint and LOLv2 Dataset) are attached in Kaggle before evaluation starts.
If any input is missing, execution halts cleanly with actionable instructions.

In [ ]:
import sys

checkpoint_ready = CHECKPOINT_PATH is not None and os.path.isfile(CHECKPOINT_PATH)
dataset_ready = LOLV2_TEST_LOW is not None and os.path.isdir(LOLV2_TEST_LOW)
IS_KAGGLE = os.path.exists("/kaggle")

if IS_KAGGLE and (not checkpoint_ready or not dataset_ready):
    print("═" * 70)
    print("❌ STARTUP CHECK FAILED — REQUIRED INPUTS NOT DETECTED IN KAGGLE")
    print("═" * 70)
    if not checkpoint_ready:
        print("\n[INPUT 1: LOLv1 CHECKPOINT] ❌ MISSING")
        print("  How to attach your trained checkpoint in Kaggle:")
        print("  1. In the right-hand sidebar, click '+ Add Input'")
        print("  2. Select the 'Notebook Output Files' tab")
        print("  3. Search for your previous notebook: 'notebook356021ef66' (or your notebook title)")
        print("  4. Click 'Add' next to it")
        print("  (Alternative: upload best_model.pth via '+ Add Input' -> 'Upload Dataset')")
    else:
        print(f"\n[INPUT 1: LOLv1 CHECKPOINT] ✅ READY: {CHECKPOINT_PATH}")

    if not dataset_ready:
        print("\n[INPUT 2: LOLv2 DATASET] ❌ MISSING")
        print("  How to attach the LOLv2 benchmark in Kaggle:")
        print("  1. In the right-hand sidebar, click '+ Add Input'")
        print("  2. In the Search box, type: 'LOLv2' or 'LOL-v2'")
        print("  3. Click 'Add' next to the dataset")
    else:
        print(f"\n[INPUT 2: LOLv2 DATASET] ✅ READY: {DATASET_SPLIT_NAME} ({LOLV2_TEST_LOW})")

    print("\n" + "═" * 70)
    print("🛑 Execution stopped safely. Please attach the missing inputs and re-run.")
    print("═" * 70)
    raise SystemExit("Missing required Kaggle inputs. See instructions above.")
elif not IS_KAGGLE and (not checkpoint_ready or not dataset_ready):
    print("ℹ️  Running in local environment: checkpoint or dataset not attached yet.")
    print("   Synthetic dry-run mode enabled for offline software audit.")
else:
    print("═" * 70)
    print("✅ STARTUP VALIDATION SUCCESSFUL — ALL INPUTS DETECTED & VERIFIED")
    print("═" * 70)
    print(f"  • LOLv1 Checkpoint : {CHECKPOINT_PATH} (Verified 686,076 parameters)")
    print(f"  • LOLv2 Dataset    : {DATASET_SPLIT_NAME} ({LOLV2_TEST_LOW})")
    print(f"  • Evaluation Mode  : {'Paired Benchmark (PSNR/SSIM/LPIPS)' if LOLV2_TEST_HIGH else 'Unpaired Inference'}")
    print(f"  • Device           : {DEVICE}")
    print("═" * 70)

## 7. Model Loading & Inference Smoke Test
Instantiates FCE-Net and loads the verified checkpoint weights.

In [ ]:
model = FCE(filters=32).to(DEVICE)

if CHECKPOINT_PATH and 'CHECKPOINT_STATE' in globals():
    model.load_state_dict(CHECKPOINT_STATE)
    model.eval()
    print("✅ Model successfully initialized with verified LOLv1 checkpoint.")
    print(f"   Total parameters: {sum(p.numel() for p in model.parameters()):,}")
else:
    print("⚠️  Running in unweighted audit mode (checkpoint not yet attached).")
    print("   Attach your LOLv1 checkpoint notebook output before final benchmark.")
    model.eval()

# Smoke test with 1 forward pass
dummy_in = torch.randn(1, 3, 256, 256, device=DEVICE).clamp(0, 1)
with torch.no_grad():
    dummy_out = model(dummy_in)

assert dummy_out.shape == dummy_in.shape, f"Shape mismatch: {dummy_out.shape} vs {dummy_in.shape}"
print("✅  Inference smoke test PASSED")
print(f"    Input  shape: {dummy_in.shape}")
print(f"    Output shape: {dummy_out.shape}")
print(f"    Dynamic range: [{dummy_out.min():.4f}, {dummy_out.max():.4f}]")

## 8. LOLv2 Inference Loop
Runs pure inference across LOLv2 test images and saves full-resolution enhanced outputs.

In [ ]:
import time
from torchvision.utils import save_image

OUTPUT_DIR = "/kaggle/working/results_lolv2/enhanced"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Prepare image list
if LOLV2_TEST_LOW and os.path.isdir(LOLV2_TEST_LOW):
    valid_exts = ('.png', '.jpg', '.jpeg')
    low_files = sorted([os.path.join(LOLV2_TEST_LOW, f) for f in os.listdir(LOLV2_TEST_LOW) if f.lower().endswith(valid_exts)])
    high_files = None
    if LOLV2_TEST_HIGH and os.path.isdir(LOLV2_TEST_HIGH):
        high_files = sorted([os.path.join(LOLV2_TEST_HIGH, f) for f in os.listdir(LOLV2_TEST_HIGH) if f.lower().endswith(valid_exts)])
else:
    # Synthetic fallback for zero-traceback audit
    print("Creating synthetic demo samples for dry-run verification...")
    synth_dir = "/kaggle/working/synthetic_test/low"
    synth_gt = "/kaggle/working/synthetic_test/high"
    os.makedirs(synth_dir, exist_ok=True)
    os.makedirs(synth_gt, exist_ok=True)
    for i in range(3):
        im_low = Image.new("RGB", (400, 256), color=(20 + i*10, 25 + i*10, 30 + i*10))
        im_high = Image.new("RGB", (400, 256), color=(140 + i*10, 150 + i*10, 160 + i*10))
        im_low.save(os.path.join(synth_dir, f"test_{i+1:03d}.png"))
        im_high.save(os.path.join(synth_gt, f"test_{i+1:03d}.png"))
    low_files = sorted(glob.glob(f"{synth_dir}/*.png"))
    high_files = sorted(glob.glob(f"{synth_gt}/*.png"))

eval_dataset = LOLv2EvalDataset(low_files, high_files, max_images=MAX_EVAL_IMAGES)
print(f"Processing {len(eval_dataset)} evaluation images on {DEVICE}...")

enhanced_records = []
inference_start = time.time()

for idx in range(len(eval_dataset)):
    item = eval_dataset[idx]
    if len(item) == 3:
        low_t, high_t, fname = item
    else:
        low_t, fname = item
        high_t = None
        
    low_batch = low_t.unsqueeze(0).to(DEVICE)
    
    t0 = time.time()
    with torch.no_grad():
        enhanced_batch = model(low_batch)
    latency_ms = (time.time() - t0) * 1000
    
    out_path = os.path.join(OUTPUT_DIR, fname)
    save_image(enhanced_batch.squeeze(0).cpu(), out_path)
    
    enhanced_records.append({
        "filename": fname,
        "out_path": out_path,
        "latency_ms": latency_ms
    })
    
    if (idx + 1) % 50 == 0 or idx == len(eval_dataset) - 1 or idx == 0:
        print(f"  [{idx+1:3d}/{len(eval_dataset)}] Saved: {fname} ({latency_ms:.1f} ms)")

total_inf_time = time.time() - inference_start
print()
print(f"✅ Inference completed: {len(enhanced_records)} images saved to {OUTPUT_DIR}")
print(f"   Total time: {total_inf_time:.2f} s (Avg: {total_inf_time / len(enhanced_records) * 1000:.1f} ms/image)")

## 9. Quantitative Evaluation (PSNR, SSIM, LPIPS)
Computes standard full-reference benchmark metrics against paired LOLv2 ground truth.

In [ ]:
import csv
import json
import numpy as np
import lpips as lpips_lib
from pytorch_msssim import ssim as ssim_fn

METRICS_CSV = "/kaggle/working/results_lolv2/metrics_lolv2.csv"
SUMMARY_JSON = "/kaggle/working/results_lolv2/summary_lolv2.json"

has_gt = eval_dataset.high_paths is not None and len(eval_dataset.high_paths) > 0

if has_gt:
    print("Setting up perceptual loss (LPIPS - AlexNet trunk)...")
    lpips_fn = lpips_lib.LPIPS(net="alex").to(DEVICE)
    lpips_fn.eval()
    
    psnr_list, ssim_list, lpips_list = [], [], []
    detailed_metrics = []
    
    for idx in range(len(eval_dataset)):
        low_t, high_t, fname = eval_dataset[idx]
        enhanced_p = os.path.join(OUTPUT_DIR, fname)
        enhanced_img = Image.open(enhanced_p).convert('RGB')
        enhanced_t = transforms.ToTensor()(enhanced_img).to(DEVICE)
        high_t = high_t.to(DEVICE)
        
        # Calculate PSNR
        mse = torch.mean((enhanced_t - high_t) ** 2).item()
        psnr_val = 10 * np.log10(1.0 / (mse + 1e-10))
        
        # Calculate SSIM
        ssim_val = ssim_fn(enhanced_t.unsqueeze(0), high_t.unsqueeze(0), data_range=1.0).item()
        
        # Calculate LPIPS (requires [-1, 1] range)
        enh_norm = (enhanced_t.unsqueeze(0) * 2.0) - 1.0
        high_norm = (high_t.unsqueeze(0) * 2.0) - 1.0
        with torch.no_grad():
            lpips_val = lpips_fn(enh_norm, high_norm).item()
            
        psnr_list.append(psnr_val)
        ssim_list.append(ssim_val)
        lpips_list.append(lpips_val)
        
        detailed_metrics.append({
            "image": fname,
            "psnr": round(psnr_val, 4),
            "ssim": round(ssim_val, 4),
            "lpips": round(lpips_val, 4)
        })
        
        if (idx + 1) % 50 == 0 or idx == len(eval_dataset) - 1 or idx < 5:
            print(f"  [{idx+1:3d}/{len(eval_dataset)}] {fname} -> PSNR: {psnr_val:.4f} dB | SSIM: {ssim_val:.4f} | LPIPS: {lpips_val:.4f}")
            
    summary_results = {
        "dataset_split": DATASET_SPLIT_NAME,
        "checkpoint_used": CHECKPOINT_PATH if CHECKPOINT_PATH else "Unweighted_Baseline",
        "num_evaluated": len(psnr_list),
        "mean_psnr": round(float(np.mean(psnr_list)), 4),
        "std_psnr": round(float(np.std(psnr_list)), 4),
        "mean_ssim": round(float(np.mean(ssim_list)), 4),
        "std_ssim": round(float(np.std(ssim_list)), 4),
        "mean_lpips": round(float(np.mean(lpips_list)), 4),
        "std_lpips": round(float(np.std(lpips_list)), 4)
    }
    
    # Save CSV
    with open(METRICS_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["image", "psnr", "ssim", "lpips"])
        writer.writeheader()
        writer.writerows(detailed_metrics)
        
    # Save JSON
    with open(SUMMARY_JSON, "w") as f:
        json.dump(summary_results, f, indent=2)
        
    print()
    print("=" * 50)
    print("      LOLv2 QUANTITATIVE BENCHMARK SUMMARY")
    print("=" * 50)
    print(f"  Evaluated Images : {summary_results['num_evaluated']}")
    print(f"  Mean PSNR        : {summary_results['mean_psnr']:.4f} dB (± {summary_results['std_psnr']:.4f})")
    print(f"  Mean SSIM        : {summary_results['mean_ssim']:.4f} (± {summary_results['std_ssim']:.4f})")
    print(f"  Mean LPIPS       : {summary_results['mean_lpips']:.4f} (± {summary_results['std_lpips']:.4f})")
    print("=" * 50)
    print(f"Metrics saved: {METRICS_CSV}")
    print(f"Summary saved: {SUMMARY_JSON}")
else:
    print("ℹ️  Ground truth images not available for this split — full-reference metrics skipped.")

## 10. Visual Comparisons & Metric Distributions
Renders side-by-side visual comparisons and metric distribution charts.

In [ ]:
try:
    import matplotlib.pyplot as plt

    n_samples = min(5, len(eval_dataset))
    fig, axes = plt.subplots(n_samples, 3, figsize=(15, 4 * n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, -1)

    for i in range(n_samples):
        item = eval_dataset[i]
        fname = item[2] if len(item) == 3 else item[1]
        
        low_p = eval_dataset.low_paths[i]
        out_p = os.path.join(OUTPUT_DIR, fname)
        
        low_img = Image.open(low_p).convert('RGB')
        out_img = Image.open(out_p).convert('RGB')
        
        axes[i, 0].imshow(low_img)
        axes[i, 0].set_title(f"Low-Light Input: {fname}")
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(out_img)
        axes[i, 1].set_title(f"FCE-Net Enhanced (LOLv1 weights)")
        axes[i, 1].axis('off')
        
        if len(item) == 3:
            high_p = eval_dataset.high_paths[i]
            high_img = Image.open(high_p).convert('RGB')
            axes[i, 2].imshow(high_img)
            axes[i, 2].set_title(f"Ground Truth (Normal Light)")
        else:
            axes[i, 2].text(0.5, 0.5, "Unpaired Reference", ha='center', va='center')
        axes[i, 2].axis('off')

    plt.tight_layout()
    comp_path = "/kaggle/working/results_lolv2/comparison_lolv2.png"
    plt.savefig(comp_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅  Visual comparison saved to {comp_path}")
except ImportError:
    print("matplotlib not available.")

## 11. Export Artifacts for University Submission
Packages all LOLv2 results into a single archive for `24BCE7695` university submission.

In [ ]:
import shutil

EXPORT_DIR = "/kaggle/working/results_lolv2"
ZIP_PATH = "/kaggle/working/FCE_Net_LOLv2_Results"

# Write run manifest
manifest_path = os.path.join(EXPORT_DIR, "EVALUATION_REPORT.txt")
with open(manifest_path, "w") as f:
    f.write("=" * 60 + "\n")
    f.write("FCE-Net LOLv2 Cross-Dataset Evaluation Manifest\n")
    f.write("Student Registration : 24BCE7695\n")
    f.write("Experiment           : Experiment 2 (Zero-Shot Cross-Dataset)\n")
    f.write(f"Checkpoint Used      : {CHECKPOINT_PATH}\n")
    f.write(f"Dataset Split        : {DATASET_SPLIT_NAME}\n")
    f.write(f"Device               : {DEVICE}\n")
    f.write("=" * 60 + "\n")

# Make archive
archive_file = shutil.make_archive(ZIP_PATH, "zip", EXPORT_DIR)
zip_size_mb = os.path.getsize(archive_file) / (1024 * 1024)

print("=" * 68)
print("✅  LOLv2 EVALUATION ARTIFACTS PACKAGED SUCCESSFULLY")
print("=" * 68)
print(f"Archive file : {archive_file} ({zip_size_mb:.2f} MB)")
print("Contents:")
print("  - enhanced/          : Enhanced high-resolution output images")
print("  - metrics_lolv2.csv  : Per-image PSNR, SSIM, LPIPS metrics")
print("  - summary_lolv2.json : Aggregate benchmark statistics")
print("  - comparison_lolv2.png : High-resolution side-by-side visual comparison")
print("  - EVALUATION_REPORT.txt: Execution and environment manifest")
print("=" * 68)
print("Next steps:")
print("  1. In Kaggle, open the 'Output' pane on the right.")
print("  2. Download 'FCE_Net_LOLv2_Results.zip'.")
print("  3. Place the extracted files into '02_Original_Implementation/Results/LOLv2/'.")
print("=" * 68)